# Notebook 05 — Outlier Effect on MSE (Batch GD)

**Session:** Linear Regression — L1, L2, and Huber Losses (Level 2)  
**Module:** 02 · Calculus and Linear Algebra

## Goal

Train a linear model with **MSE** on `data_outlier.csv`, which contains one additional
outlier sample (a property with a disproportionately high price relative to its area).
The quadratic penalty in MSE causes the fitted line to be pulled toward the outlier.

**Learning objectives:**
- Demonstrate that MSE is sensitive to outliers.
- Motivate the use of MAE and Huber loss for robust regression.
- Visually compare the fitted line with and without the outlier.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

## 1  Load Data with Outlier

In [ ]:
def get_column(data: list, index: int) -> list:
    return [row[index] for row in data]


data = np.genfromtxt("../data/data_outlier.csv", delimiter=",").tolist()
x_data = get_column(data, 0)
y_data = get_column(data, 1)
N = len(x_data)

print(f"areas  : {x_data}")
print(f"prices : {y_data}")
print(f"N      : {N}  (original = 4, +1 outlier)")

## 2  Model Functions

In [ ]:
def predict(x: float, w: float, b: float) -> float:
    return w * x + b


def gradient_mse(y_hat: float, y: float, x: float) -> tuple[float, float]:
    dw = 2 * x * (y_hat - y)
    db = 2 * (y_hat - y)
    return dw, db


def update_weight(w: float, b: float, lr: float, dw: float, db: float) -> tuple[float, float]:
    return w - lr * dw, b - lr * db

## 3  Batch GD Training Loop

In [ ]:
w: float = -0.34
b: float = 0.04
lr: float = 0.01
epoch_max: int = 20

epoch_losses: list[float] = []

for _epoch in range(epoch_max):
    dw_total = 0.0
    db_total = 0.0
    loss_total = 0.0

    for i in range(N):
        x = x_data[i]
        y = y_data[i]

        y_hat = predict(x, w, b)
        loss_total += (y_hat - y) ** 2

        dw, db = gradient_mse(y_hat, y, x)
        dw_total += dw
        db_total += db

    epoch_losses.append(loss_total / N)
    w, b = update_weight(w, b, lr, dw_total / N, db_total / N)

print(f"Final parameters  →  w = {w:.4f}, b = {b:.4f}")

## 4  Visualise the Outlier Effect

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(epoch_losses)
plt.xlabel("Epoch")
plt.ylabel("Mean MSE loss")
plt.title("MSE Loss — Dataset with Outlier")
plt.tight_layout()
plt.show()

In [ ]:
# Identify the outlier (last point in dataset)
regular_x = x_data[:-1]
regular_y = y_data[:-1]
outlier_x = [x_data[-1]]
outlier_y = [y_data[-1]]

x_line = list(range(2, 8))
y_line = [predict(x, w, b) for x in x_line]

plt.figure(figsize=(6, 4))
plt.scatter(regular_x, regular_y, label="Regular samples")
plt.scatter(outlier_x, outlier_y, color="red", marker="x", s=100, label="Outlier")
plt.plot(x_line, y_line, "g-", label=f"MSE fit: y = {w:.3f}x + {b:.3f}")
plt.xlabel("Area (×100 m²)")
plt.ylabel("Price (Tael)")
plt.title("MSE is pulled toward the outlier")
plt.legend()
plt.tight_layout()
plt.show()

### Observation

The fitted line is pulled noticeably toward the outlier because MSE assigns a quadratic
penalty to large residuals.  Try replacing `gradient_mse` with `gradient_mae` from Notebook
04 to see how MAE handles the outlier more gracefully.